# पाठ १८: क्रिप्टोग्राफिक रसिदहरु मार्फत AI एजेन्टहरुलाई सुरक्षित पार्दै

## व्यावहारिक नोटबुक

यो नोटबुकले चार अभ्यासमार्फत अघि बढ्छ:

1. एजेण्ट उपकरण कलको लागि **तपाईंको पहिलो रसिदमा हस्ताक्षर गर्नुहोस्** र त्यसलाई प्रमाणित गर्नुहोस्।
2. रसिदसँग **छेड़छाड़ गर्नुहोस्** र प्रमाणिकरण विफल हुँदै हेर्नुहोस्।
3. **तीन रसिदको श्रृंखला बनाउनुहोस्** र श्रृंखलाको अखण्डता पुष्टि गर्नुहोस्।
4. हरेक क्रियाकलापले रसिद उत्सर्जन गर्ने गरी **Microsoft Agent Framework उपकरण कललाई र्याप गर्नुहोस्**।

सबै क्रिप्टोग्राफिक प्रिमिटिभहरू राम्रोसँग मर्मत राखिएका पुस्तकालयहरूबाट आयात गरिएको छ (`pynacl` Ed25519 का लागि, `jcs` RFC 8785 कानोनिकल JSON का लागि, Python मानक पुस्तकालयको `hashlib` SHA-256 का लागि)। रसिदको तर्क आफैं सरल Python हो जुन तपाईं पढ्न र संशोधन गर्न सक्नुहुन्छ।

कोषहरू क्रमबद्ध रूपमा चलाउनुहोस्। प्रत्येक भाग छोटो र स्व-सम्पूर्ण छ।


## सेटअप

यी दुई निर्भरता स्थापना गर्नुहोस्। दुवैले अनुमति दिने लाइसेन्सहरू छन् (Apache-2.0 / MIT)।


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## सहायक युटिलिटीज

यी दुई सहायकहरूले आधार64url एन्कोडिङ (प्याडिङ बिना) र मनमानी वस्तुहरूको SHA-256 हैशिङ नियन्त्रित गर्छन्। तिनीहरूले बाँकी नोटबुकलाई रसीद तर्कमा केन्द्रित राख्छन्।


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## खण्ड १: तपाईंको पहिलो रसिदमा हस्ताक्षर गर्नुहोस्

कल्पना गर्नुहोस् कि हाम्रो **Contoso Travel** एजेन्टले एउटा ग्राहकका लागि सिड्नीबाट लस एन्जेलस सम्मको उडानहरू खोज्यो। हामी यो टुल कललाई हस्ताक्षरित रसिदको रूपमा रेकर्ड गर्न चाहन्छौं ताकि भविष्यमा कुनै परीक्षकले यसलाई हामीमाथि विश्वास नगरी सत्यापित गर्न सकोस्।

### चरण १.१: हस्ताक्षर गर्ने कुञ्जी बनाउनुहोस्

उत्पादनमा, एजेन्टको हस्ताक्षर गर्ने कुञ्जी हार्डवेयर सुरक्षा मोड्युल (HSM), Azure Key Vault, वा समान सुरक्षीत स्टोरमा राखिन्छ। यस पाठका लागि हामी स्मृतिमा नया कुञ्जी निर्माण गर्छौं।


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### चरण 1.2: रसिद पेलोड बनाउनुहोस्

पेलोडले हामीले रसिदले प्रमाणित गर्न चाहेका सबै कुरा समावेश गर्दछ: को क्रियाशील भयो, कुन उपकरणले, के कस्ता तर्कहरूसँग, के फर्कियो, कुन नीतिमा, र कहिले। हामीले तर्कहरू र नतिजालाई इनलाइन समावेश नगरी त्यसको हैश गर्छौं ताकि रसिदले संवेदनशील सामग्री नफैलाओस्।


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### चरण 1.3: रिसीटमा हस्ताक्षर गर्नुहोस् र जम्मा गर्नुहोस्

तीन चरणहरू:

1. JCS को प्रयोग गरेर पेलोड क्यानोनिकलाइज गर्नुहोस् ताकि दुई कार्यान्वयनहरूले उस्तै तर्कसंगत रिसीट उत्पन्न गर्दा बाइट-एकैजस्तो बाइटहरू उत्पादन गर्न सकोस्।
2. क्यानोनिकल बाइटहरूलाई SHA-256 सँग ह्यास गर्नुहोस्।
3. Ed25519 निजी कुञ्जीसँग ह्यासमा हस्ताक्षर गर्नुहोस्।

त्यसपछि हस्ताक्षर मूल पेलोडमा जोडिन्छ र अन्तिम रिसीट तयार हुन्छ।


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### चरण १.४: रसिदको सत्यापन गर्नुहोस्

सत्यापन प्रक्रियालाई उल्टो गर्दछ। हामी हस्ताक्षर हटाउँछौं, क्यानोनिकल ह्यास पुन: गणना गर्छौं, र रसिदमा रहेको सार्वजनिक कुञ्जीसँग हस्ताक्षर जाँच गर्छौं।

यो सत्यापन गर्ने अडिटरलाई हामीबाट केही चाहिँदैन सिवाय रसिदको। कुनै सेवा कल गर्नुपर्ने, कुनै कुञ्जी निर्देशिका सोध्नुपर्ने, वा कुनै भरोसा आवश्यक छैन।


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

तपाईंले `Receipt is valid: True` देख्नुपर्छ। एजेन्टले यसको पहिलो क्रिप्टोग्राफिक रूपमा हस्ताक्षरित अडिट रेकर्ड उत्पादन गरेको छ।


## अनुभाग २: रसिदमा छेउछाउ गर्नुहोस्

रसिदहरूको सम्पूर्ण उद्देश्य तिनीहरू छेउछाउ-स्पष्ट हुन्। आउनुहोस् यसलाई प्रमाणित गरौं।

हामी रसिदको ठीक एउटा अक्षर परिमार्जन गर्नेछौं र प्रमाणीकरण असफल हुन हेर्नेछौं।


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### के भयो अहिले?

जब हामीले `policy_id` परिवर्तन गर्यौं, क्यानोनिकल बाइटहरू परिवर्तन भए। ती बाइटहरूको SHA-256 ह्यास परिवर्तन भयो। त्यस ह्यासमाथि आधारित हस्ताक्षर अब नयाँ ह्याससँग मेल खाँदैन। भेरिफिकेसन सही रूपमा `False` फर्काउँछ।

प्राप्तिको कुनै पनि फिल्ड परिवर्तन गरेर पनि प्रमाणित गर्न सकिन्न, जबसम्म आक्रमणकारीसँग निजी कुञ्जी हुँदैन। निजी कुञ्जी कुञ्जी तलामा हुन्छ र सार्वजनिक कुञ्जी प्रकाशित गरिएको छ भने, छेड़छाड़ लुकाउन असम्भव हुन्छ।

आफैं प्रयास गर्नुहोस्: माथिको सेलमा `tool_name` वा `agent_id` वा `timestamp` परिवर्तन गरेर पुनः चलाउनुहोस्। प्रत्येक परिवर्तनले अमान्य प्राप्ति उत्पन्न गर्छ।


## खण्ड ३: रसिदहरूलाई सँगै जोड्नुहोस्

एउटा रसिदले एउटा क्रियालाई मात्र सुरक्षा गर्छ। अधिकांश एजेन्टहरूले धेरै क्रियाहरू गर्छन्। सम्पूर्ण अनुक्रमलाई छेड़छाड़-सिद्ध बनाउन, हामी प्रत्येक रसिदलाई त्यस अघिल्लो रसिदको ह्यास नयाँ रसिदको पेलोडमा समावेश गरेर जोड्छौं।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

यदि कसैले रसिद हटाउँछ वा पुनःक्रमबद्ध गर्छ भने, चेन ठीक त्यस बुँदामा टुट्छ। कुनै पनि पछि आउने रसिदको प्रमाणीकरण असफल हुन्छ किनभने यसको `previous_receipt_hash` अब यसको पूर्ववर्तीको वास्तविक ह्याससँग मेल खाँदैन।


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

अब मध्यम रसिदमा छेड़छाड़ गरेर श्रृंखला तोड्नुहोस् र पुन: प्रमाणीकरण गर्नुहोस्। छेड़छाड़ गरिएको रसिद यसको हस्ताक्षर जाँचमा असफल हुन्छ, र अर्को रसिद यसको श्रृंखला-लिङ्क जाँचमा असफल हुन्छ (किनभने यसको `previous_receipt_hash` अब परिमार्जित मध्यम रसिदको ह्याससँग मेल खाँदैन)।


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

रसिद 0 अझै प्रमाणित हुन्छ (यो परिवर्तन गरिएको छैन र निर्भर हुनुपर्ने कुनै पूर्ववर्ती छैन)। रसिद 1 ले यसको हस्ताक्षर जाँचमा असफल हुन्छ किनकि हामीले `tool_args_hash` परिवर्तन गरेका छौं। रसिद 2 ले यसको चेन-लिंक जाँच असफल हुन्छ किनभने यसको `previous_receipt_hash` मूल (अब परिवर्तन गरिएको) रसिद 1 विरुद्ध गणना गरिएको थियो।

यदि आक्रमणकारीले परिवर्तन गरिएको रसिद 1 पुनः हस्ताक्षर गरे पनि (जुन निजी कुञ्जी बिना गर्न सक्दैन), रसिद 2 मा चेन-लिंक असमानता फेरि छलफल प्रकट गर्नेछ। परिवर्तन लुकाउनको लागि, आक्रमणकारीले परिवर्तन बिन्दुबाट सबै रसिदहरू पुनः हस्ताक्षर गर्नुपर्ने हुन्छ, जुन निजी कुञ्जीको स्वामित्व आवश्यक पर्छ।


## भाग ४: एउटा एजेन्ट उपकरण कललाई रसीद हस्ताक्षरसँग जडान गर्ने

वास्तविक परिनियोजनमा, तपाईं चाहनुहुन्न कि प्रत्येक एजेन्ट लेखकले `make_receipt` कल गर्न सम्झनु पाओस्। तपाईं चाहनुहुन्छ कि हरेक उपकरण कलमा रसीद हस्ताक्षर स्वचालित होस्।

यहाँ सबैभन्दा सरल ढाँचा छ: एउटा रैपर वर्ग जसले कुनै पनि कल गर्न सकिने उपकरण कार्यलाई लिन्छ र यसको रसीद जारी गर्ने संस्करण फर्काउँछ। यसले कुनै पनि एजेन्ट फ्रेमवर्कसँग अनुकूल हुन्छ, जसमा Microsoft एजेन्ट फ्रेमवर्क (`agent_framework.foundry`) पनि समावेश छ।

यदि तपाईंले Microsoft Foundry प्रोजेक्ट सेटअप गर्नुभएको छैन भने, तलको स्थानीय मोकले पनि यो ढाँचालाई प्रदर्शन गर्दछ।


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### माइक्रोसफ्ट एजेन्ट फ्रेमवर्कसँग एकीकरण

माथि `ReceiptedTool` र्‍यापर फ्रेमवर्क-स्वतंत्र छ। माइक्रोसफ्ट एजेन्ट फ्रेमवर्कसहित बनाइएको एजेन्ट भित्र यसलाई प्रयोग गर्नका लागि, र्‍याप गरिएको फलामलाई उपकरणको रूपमा दर्ता गर्नुहोस्। एउटा स्केच (तपाईंले मॉकलाई वास्तविक माइक्रोसफ्ट फाउन्ड्री उपकरण दर्तासँग प्रतिस्थापन गर्नुहुनेछ):

```python
# समाकलन आकृति देखाउने छद्मकोड।
# आयात गर्नुहोस् os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="तपाईं एक Contoso Travel एजेन्ट हुनुहुन्छ ..."
#     tools=[receipted_lookup],   # प्याक गरिएको उपकरण, कच्चा कार्य होइन
# )
# response = agent.run("सिड्नीबाट लस एन्जेलस जान जाने फ्लाइटहरू जुन महिनामा फेला पार्नुहोस्।")
#
# # चलाएपछि, एजेन्टले गरेको प्रत्येक उपकरण कलसँग एक हस्ताक्षरित रसीद हुन्छ:
# audit_chain = receipted_lookup.receipts
```

एजेन्ट फ्रेमवर्कले रसिदहरूबारे केही जान्न आवश्यक छैन। रसिद हस्ताक्षर उपकरण वरिपरि र्‍याप गरिएको छ, फ्रेमवर्कभित्र जोडिएको छैन। यसरी तपाईं एजेन्ट पुनःलेखन नगरी विद्यमान एजेन्ट कोडमा स्रोत थप्न सक्नुहुन्छ।


## पुनरावलोकन र विस्तार चुनौती

तपाईंसँग छ:

- Ed25519 कुञ्जी जोडी सिर्जना गर्नुभयो।
- एक एजेन्ट उपकरण कलको लागि रसीद निर्माण गर्नु भयो र हस्ताक्षर गर्नुभयो।
- केवल सार्वजनिक कुञ्जी प्रयोग गरेर रसीद अफलाइन प्रमाणित गर्नुभयो।
- रसीदमा छेडछाड गर्नुभयो र प्रमाणिकरण विफल भएपछि अवलोकन गर्नुभयो।
- तीन रसीदहरूको ह्यास-श्रृंखला अनुक्रम तयार गर्नुभयो।
- श्रृंखलाको बीचमा छेडछाड गर्नुभयो र हस्ताक्षर विफलता र श्रृंखला-लिंक विफलता दुबै देख्नुभयो।
- एक उपकरण कार्यलाई स्वत: रसीद हस्ताक्षर गर्ने तरिकाले आवरण गर्नुभयो।

**विस्तार चुनौती।** रसीद स्किमामा `request_id` क्षेत्र (वितरित ट्रेसिङका लागि UUID) थप्नुहोस्। `make_receipt` लाई अपडेट गर्नुहोस् ताकि यो समावेश होस्, र रसीदहरू अझै अन्त-देखि-अन्त प्रमाणित भइरहेको छ पुष्टि गर्नुहोस्। त्यसपछि हस्ताक्षर गरेपछि क्षेत्रलाई संशोधन गरी प्रमाणिकरण विफल भएको कुरा पुष्टि गर्नुहोस्। यसले तपाईंलाई क्यानोनिकल एन्कोडिङको प्रत्येक बाइटले कसरी हस्ताक्षरमा योगदान पुर्‍याउँछ भन्‍ने कुरा आन्तरिक बनाउन बाध्य बनाउँछ।

**महत्वपूर्ण सीमा।** रसीदहरूले तीन कुरा मात्र प्रमाणित गर्छन् र तीनै कुरा मात्र: आवंटन (यो कुञ्जीले यो सामग्रीमा हस्ताक्षर गरेको छ), अखण्डता (हस्ताक्षरपछिको सामग्री परिवर्तन भएको छैन), र क्रम (यो रसीद त्यो रसीदपछि आएको हो)। यसले प्रमाणित गर्दैन कि एजेन्टको कार्य सही थियो, `policy_id` मा नाम गरिएको नीति वास्तवमा मूल्याङ्कन गरिएको थियो, वा एजेन्टले सबै नियमहरू पालन गर्‍यो। रसीदहरू आधार हुन्। प्रशासन तपाईँले माथि निर्माण गर्ने प्रणाली हो।

त्यो सीमालाई ध्यानमा राख्दै पाठ पुनः पढ्नुस्। रसीदसँग सबभन्दा सामान्य गल्ती टोलीहरूले गर्ने कुरा भनेको “हामीसँग रसीदहरू छन्” भनेको “हामी प्रबन्धित छौं” हो भनेर मान्नु हो। त्यस्तो होइन। रसीदहरूले एजेन्ट व्यवहारलाई अडिटयोग्य बनाउँछ। यसले त्यसलाई सही बनाउँदैन।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
